In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

DATA_DIR = Path(
    "/Users/manyamoonat/pbl sem5/Euphorbia_Augmented_3000"
)

IMG_SIZE = (128, 128)
BATCH_SIZE = 16
SEED = 42
EPOCHS = 30

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Dataset exists:", DATA_DIR.exists())

for folder in sorted(DATA_DIR.iterdir()):
    if folder.is_dir():
        print(folder.name)

TensorFlow: 2.16.2
Dataset exists: True
.ipynb_checkpoints
diseased
healthy


In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.30,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True
)

class_names = train_ds.class_names

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    class_names=class_names,
    shuffle=True
)

print("Class order:", class_names)

Found 3214 files belonging to 2 classes.
Using 2250 files for training.
Found 3214 files belonging to 2 classes.
Using 964 files for validation.
Class order: ['diseased', 'healthy']


2026-10-23 09:48:29.812325: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-10-23 09:48:29.812356: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-10-23 09:48:29.812359: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-10-23 09:48:29.812377: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-23 09:48:29.812390: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

def prepare_dataset(ds, training=False):
    ds = ds.map(
        lambda images, labels: (
            tf.cast(images, tf.float32) / 255.0,
            labels
        ),
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.shuffle(500, seed=SEED)

    return ds.prefetch(AUTOTUNE)

train_ds = prepare_dataset(train_ds, training=True)
val_ds = prepare_dataset(val_ds)

print("Datasets prepared.")

Datasets prepared.


In [4]:
from tensorflow.keras import layers, models, regularizers

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Augmentation applied during training only
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),

    layers.Conv2D(
        32, 3, padding="same",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.20),

    layers.Conv2D(
        64, 3, padding="same",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(
        128, 3, padding="same",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.30),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 110,785 (432.75 KB)

 Trainable params: 110,337 (431.00 KB)

 Non-trainable params: 448 (1.75 KB)

In [6]:
OUTPUT_DIR = Path("custom_cnn_augmented_results")
OUTPUT_DIR.mkdir(exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(OUTPUT_DIR / "best_custom_cnn.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/30


2026-10-23 10:11:43.967630: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:17: Filling up shuffle buffer (this may take a while): 77 of 500


  1/141 ━━━━━━━━━━━━━━━━━━━━ 43:09 18s/step - accuracy: 0.7500 - loss: 0.9440 - precision: 0.7273 - recall: 0.8889

2026-10-23 10:11:52.227291: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6509 - loss: 0.7192 - precision: 0.6975 - recall: 0.7667
Epoch 1: val_loss improved from None to 0.74092, saving model to custom_cnn_augmented_results/best_custom_cnn.keras
141/141 ━━━━━━━━━━━━━━━━━━━━ 36s 125ms/step - accuracy: 0.6498 - loss: 0.7078 - precision: 0.6913 - recall: 0.7690 - val_accuracy: 0.6400 - val_loss: 0.7409 - val_precision: 0.6275 - val_recall: 0.9862 - learning_rate: 7.5000e-05
Epoch 2/30


...

141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.7123 - loss: 0.6470 - precision: 0.7451 - recall: 0.8045
Epoch 10: val_loss did not improve from 0.68188
141/141 ━━━━━━━━━━━━━━━━━━━━ 44s 135ms/step - accuracy: 0.6956 - loss: 0.6538 - precision: 0.7282 - recall: 0.7988 - val_accuracy: 0.6058 - val_loss: 0.9531 - val_precision: 0.6038 - val_recall: 1.0000 - learning_rate: 3.7500e-05
Epoch 11/30


2026-10-23 10:18:13.875881: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:17: Filling up shuffle buffer (this may take a while): 61 of 500


  1/141 ━━━━━━━━━━━━━━━━━━━━ 53:47 23s/step - accuracy: 0.6875 - loss: 0.7956 - precision: 0.6000 - recall: 0.8571

2026-10-23 10:18:26.741470: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6878 - loss: 0.6628 - precision: 0.7186 - recall: 0.7939
Epoch 11: val_loss did not improve from 0.68188

Epoch 11: ReduceLROnPlateau reducing learning rate to 1.8750000890577212e-05.
141/141 ━━━━━━━━━━━━━━━━━━━━ 41s 129ms/step - accuracy: 0.7000 - loss: 0.6524 - precision: 0.7276 - recall: 0.8120 - val_accuracy: 0.6037 - val_loss: 0.9668 - val_precision: 0.6025 - val_recall: 1.0000 - learning_rate: 3.7500e-05
Epoch 11: early stopping
Restoring model weights from the end of the best epoch: 5.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Custom CNN Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Custom CNN Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=300)
plt.show()

In [ ]:
best_model = tf.keras.models.load_model(
    OUTPUT_DIR / "best_custom_cnn.keras"
)

best_model.save(OUTPUT_DIR / "custom_cnn_final.keras")

with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(
        {
            key: [float(v) for v in values]
            for key, values in history.history.items()
        },
        f,
        indent=2
    )

print("Model and history saved to:", OUTPUT_DIR.resolve())

In [ ]:
TEST_DIR = Path("/Users/manyamoonat/pbl sem5/Euphorbia_Split/test")

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    class_names=class_names,
    shuffle=False
)

test_ds = prepare_dataset(test_ds)

test_results = best_model.evaluate(test_ds, return_dict=True)

print("\nOriginal test-set results:")
for name, value in test_results.items():
    print(f"{name}: {value:.4f}")

In [ ]:
y_true = np.concatenate([
    labels.numpy().reshape(-1)
    for images, labels in test_ds
]).astype(int)

y_prob = best_model.predict(test_ds, verbose=1).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=class_names,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Custom CNN — Original Test Set")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=300)
plt.show()